# Uncertainty-Aware Object Detection for Vision-Based Driver Assistance Systems

**Paper Implementation**: FDEN + PCTAM + EDD on nuScenes Dataset  
**Dataset**: [nuScenes on Kaggle](https://www.kaggle.com/datasets/mitanshuchakrawarty/nuscenes)  
**Authors**: Kedar Kothari, Dr. Arpita Shah — CHARUSAT University

---

### Architecture Overview
1. **BEV Representation Learning** — Multi-camera feature lifting to Bird's Eye View
2. **FDEN** — Feature Distribution Estimation Network (aleatoric uncertainty)
3. **PCTAM** — Probabilistic Cross-Temporal Attention Module (uncertainty-guided temporal fusion)
4. **EDD** — Evidential Detection Decoder (3D bbox + epistemic uncertainty)

> ⚠️ Training is disabled by default. Set `TRAIN = True` to enable.


## 0. Setup & Installs

In [ ]:
# Install required packages
!pip install nuscenes-devkit torch torchvision einops --quiet

import os
import json
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from einops import rearrange
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {DEVICE}")

# ── Config ──────────────────────────────────────────────────────────────────
TRAIN        = False        # Set True to run training loop
NUM_EPOCHS   = 10
BATCH_SIZE   = 2
LR           = 1e-4
NUM_CAMERAS  = 6            # nuScenes: CAM_FRONT, CAM_FRONT_RIGHT, CAM_FRONT_LEFT,
                            #           CAM_BACK, CAM_BACK_LEFT, CAM_BACK_RIGHT
NUM_CLASSES  = 10           # nuScenes object categories
TEMPORAL_T   = 3            # Number of historical frames
BEV_H        = 50           # BEV grid height (cells)
BEV_W        = 50           # BEV grid width  (cells)
BEV_C        = 256          # BEV feature channels
IMG_H        = 256
IMG_W        = 704

NUSCENES_CLASSES = [
    'car', 'truck', 'bus', 'trailer', 'construction_vehicle',
    'pedestrian', 'motorcycle', 'bicycle', 'traffic_cone', 'barrier'
]

## 1. Kaggle Dataset Setup

In [ ]:
# ── Option A: Mount Google Drive and point to nuScenes root ─────────────────
# from google.colab import drive
# drive.mount('/content/drive')
# NUSCENES_ROOT = '/content/drive/MyDrive/nuscenes'

# ── Option B: Download directly via Kaggle API ───────────────────────────────
# 1. Upload your kaggle.json to Colab
# from google.colab import files
# files.upload()   # upload kaggle.json
# !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !kaggle datasets download -d mitanshuchakrawarty/nuscenes --unzip -p /content/nuscenes
# NUSCENES_ROOT = '/content/nuscenes'

# ── For demo / dry-run without the actual dataset ───────────────────────────
NUSCENES_ROOT = '/content/nuscenes'   # change this path after downloading

USE_MOCK = not os.path.exists(NUSCENES_ROOT)  # auto-detect missing dataset
if USE_MOCK:
    print("⚠️  Dataset not found — running in MOCK mode (random tensors).")
    print("    Set NUSCENES_ROOT to your nuScenes directory to use real data.")
else:
    print(f"✅  nuScenes root found at: {NUSCENES_ROOT}")

## 2. nuScenes Dataset Loader

In [ ]:
class NuScenesDataset(Dataset):
    """
    Loads multi-camera images and 3D bounding box annotations from nuScenes.
    Returns temporal sequences of length TEMPORAL_T+1 per sample.
    """
    CAM_NAMES = [
        'CAM_FRONT', 'CAM_FRONT_RIGHT', 'CAM_FRONT_LEFT',
        'CAM_BACK', 'CAM_BACK_LEFT', 'CAM_BACK_RIGHT'
    ]

    def __init__(self, root, split='train', temporal_t=TEMPORAL_T,
                 img_h=IMG_H, img_w=IMG_W):
        self.root       = Path(root)
        self.split      = split
        self.temporal_t = temporal_t
        self.transform  = transforms.Compose([
            transforms.Resize((img_h, img_w)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225]),
        ])
        self._build_index()

    def _build_index(self):
        """Build list of valid (scene, keyframe_idx) pairs."""
        v1_json = self.root / 'v1.0-mini' / 'sample.json'
        if not v1_json.exists():
            raise FileNotFoundError(
                f"nuScenes metadata not found at {v1_json}. "
                "Please download the dataset first."
            )
        with open(v1_json) as f:
            self.samples = json.load(f)
        print(f"Loaded {len(self.samples)} samples from nuScenes ({self.split}).")

    def __len__(self):
        return max(0, len(self.samples) - self.temporal_t)

    def _load_cameras(self, sample_token):
        """Load 6 camera images for a given sample token."""
        imgs = []
        sd_json = self.root / 'v1.0-mini' / 'sample_data.json'
        with open(sd_json) as f:
            sample_data = json.load(f)
        cam_data = {d['channel']: d for d in sample_data
                    if d['sample_token'] == sample_token
                    and d['channel'] in self.CAM_NAMES
                    and d['is_key_frame']}
        for cam in self.CAM_NAMES:
            if cam in cam_data:
                img_path = self.root / cam_data[cam]['filename']
                img = Image.open(img_path).convert('RGB')
            else:
                img = Image.new('RGB', (IMG_W, IMG_H))
            imgs.append(self.transform(img))
        return torch.stack(imgs)   # (6, 3, H, W)

    def _load_boxes(self, sample_token):
        """Load 3D bounding box annotations for a sample."""
        ann_json = self.root / 'v1.0-mini' / 'sample_annotation.json'
        cat_json = self.root / 'v1.0-mini' / 'category.json'
        with open(ann_json) as f:
            annotations = json.load(f)
        with open(cat_json) as f:
            categories = {c['token']: c['name'] for c in json.load(f)}
        boxes, labels = [], []
        for ann in annotations:
            if ann['sample_token'] != sample_token:
                continue
            cat_name = ann.get('category_name', '')
            label = next((i for i, c in enumerate(NUSCENES_CLASSES)
                          if c in cat_name), -1)
            if label == -1:
                continue
            t = ann['translation']  # [x, y, z]
            s = ann['size']         # [w, l, h]
            r = ann['rotation']     # quaternion
            boxes.append(t + s + r)
            labels.append(label)
        if boxes:
            return (torch.tensor(boxes,  dtype=torch.float32),
                    torch.tensor(labels, dtype=torch.long))
        return (torch.zeros(1, 10), torch.zeros(1, dtype=torch.long))

    def __getitem__(self, idx):
        # Temporal window: [idx, idx+1, ..., idx+T]
        frame_imgs, frame_boxes, frame_labels = [], [], []
        for t in range(self.temporal_t + 1):
            token = self.samples[idx + t]['token']
            frame_imgs.append(self._load_cameras(token))
            b, l = self._load_boxes(token)
            frame_boxes.append(b)
            frame_labels.append(l)
        return {
            'images': torch.stack(frame_imgs),   # (T+1, 6, 3, H, W)
            'boxes' : frame_boxes,               # list of (N, 10)
            'labels': frame_labels,              # list of (N,)
        }


# ── Mock Dataset (used when nuScenes files are absent) ──────────────────────
class MockNuScenesDataset(Dataset):
    """Generates random tensors matching nuScenes shapes for architecture testing."""
    def __init__(self, length=50, temporal_t=TEMPORAL_T):
        self.length     = length
        self.temporal_t = temporal_t

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        T = self.temporal_t + 1
        return {
            'images': torch.randn(T, NUM_CAMERAS, 3, IMG_H, IMG_W),
            'boxes' : [torch.rand(3, 10) for _ in range(T)],
            'labels': [torch.randint(0, NUM_CLASSES, (3,)) for _ in range(T)],
        }


# ── Instantiate ─────────────────────────────────────────────────────────────
if USE_MOCK:
    dataset = MockNuScenesDataset(length=100)
else:
    dataset = NuScenesDataset(NUSCENES_ROOT, split='train')

dataloader = DataLoader(dataset, batch_size=BATCH_SIZE,
                        shuffle=True, num_workers=0,
                        collate_fn=lambda x: x)   # variable-length boxes
print(f"Dataset size: {len(dataset)} samples")

## 3. BEV Representation Learning (Lift-Splat)

In [ ]:
class ImageBackbone(nn.Module):
    """ResNet-50 backbone, returns multi-scale features at 1/8 resolution."""
    def __init__(self, out_channels=BEV_C):
        super().__init__()
        resnet = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        # Use layers up to layer3 (stride 8)
        self.encoder = nn.Sequential(
            resnet.conv1, resnet.bn1, resnet.relu, resnet.maxpool,
            resnet.layer1, resnet.layer2, resnet.layer3
        )
        self.proj = nn.Conv2d(1024, out_channels, 1)

    def forward(self, x):
        """x: (B*T*N_cam, 3, H, W)  →  (B*T*N_cam, C, H/8, W/8)"""
        return self.proj(self.encoder(x))


class LiftSplatBEV(nn.Module):
    """
    Simplified Lift-Splat view transformer (§III-B of the paper).
    Projects multi-camera features to a top-down BEV grid.

    Full Lift-Splat uses predicted depth distributions; this implementation
    uses a learned spatial transformer for efficiency in a single-GPU Colab.
    """
    def __init__(self, in_channels=BEV_C, bev_h=BEV_H, bev_w=BEV_W,
                 n_cameras=NUM_CAMERAS):
        super().__init__()
        self.bev_h     = bev_h
        self.bev_w     = bev_w
        self.n_cameras = n_cameras
        # Depth prediction head (D discrete depth bins)
        self.D = 41    # depth bins
        self.depth_head = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels, self.D, 1)
        )
        # BEV pooling (pool camera features into BEV cells)
        self.bev_pool = nn.AdaptiveAvgPool2d((bev_h, bev_w))
        # Feature refinement after splatting
        self.bev_conv = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, 3, padding=1),
            nn.BatchNorm2d(in_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, cam_feats):
        """
        cam_feats: (B, N_cam, C, h, w)
        Returns: bev_feat (B, C, bev_h, bev_w)
        """
        B, N, C, h, w = cam_feats.shape
        # Depth softmax weights
        cam_flat = cam_feats.view(B * N, C, h, w)
        depth_w  = self.depth_head(cam_flat).softmax(dim=1)          # (BN, D, h, w)
        # Weighted feature voxel (average over depth bins)
        feat_3d  = (depth_w.unsqueeze(2) * cam_flat.unsqueeze(1)).mean(1)  # (BN, C, h, w)
        # Pool to BEV resolution and average across cameras
        bev_per_cam = self.bev_pool(feat_3d).view(B, N, C, self.bev_h, self.bev_w)
        bev_feat    = bev_per_cam.mean(dim=1)                        # (B, C, bev_h, bev_w)
        return self.bev_conv(bev_feat)


# Quick shape test
with torch.no_grad():
    _backbone = ImageBackbone().to(DEVICE)
    _bev_enc  = LiftSplatBEV().to(DEVICE)
    _dummy    = torch.randn(2, NUM_CAMERAS, 3, IMG_H, IMG_W).to(DEVICE)
    _flat     = _dummy.view(2 * NUM_CAMERAS, 3, IMG_H, IMG_W)
    _feats    = _backbone(_flat).view(2, NUM_CAMERAS, BEV_C, IMG_H//8, IMG_W//8)
    _bev      = _bev_enc(_feats)
    print(f"Backbone out : {_feats.shape}")
    print(f"BEV feat out : {_bev.shape}  — expected (2, {BEV_C}, {BEV_H}, {BEV_W})")

## 4. FDEN — Feature Distribution Estimation Network

Models each BEV spatial location as `N(μ, σ²)`.  
The variance map σ² captures **aleatoric uncertainty** (Eq. 5–6 in paper).

In [ ]:
class FDEN(nn.Module):
    """
    Feature Distribution Estimation Network (§III-C).

    Input : B_t  — BEV feature map (B, C, H, W)
    Output: mu   — feature mean     (B, C, H, W)
            sigma2 — spatial variance (B, 1, H, W)  [aleatoric uncertainty]
    """
    def __init__(self, channels=BEV_C):
        super().__init__()
        # Mean branch — refines the feature representation
        self.mean_branch = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1),
            nn.BatchNorm2d(channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, 1),
        )
        # Variance branch — lightweight head estimating per-location uncertainty
        self.var_branch = nn.Sequential(
            nn.Conv2d(channels, channels // 4, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels // 4, 1, 1),
            nn.Softplus(),   # ensures σ² > 0
        )

    def forward(self, bev_feat):
        mu     = self.mean_branch(bev_feat)
        sigma2 = self.var_branch(bev_feat)    # (B, 1, H, W)
        return mu, sigma2

    @staticmethod
    def reparameterise(mu, sigma2):
        """Sample feature from N(mu, sigma²) using reparametrisation trick."""
        eps = torch.randn_like(mu)
        return mu + eps * sigma2.sqrt()


# Visualise uncertainty map
def visualise_uncertainty_map(sigma2, title='FDEN — Feature Uncertainty Map (σ²)'):
    """sigma2: (1, 1, H, W) tensor"""
    umap = sigma2[0, 0].detach().cpu().numpy()
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(umap, cmap='RdYlGn_r', interpolation='nearest')
    plt.colorbar(im, ax=ax, label='Variance (σ²)')
    ax.set_title(title)
    ax.set_xlabel('BEV Width (W-axis)')
    ax.set_ylabel('BEV Height (H-axis)')
    plt.tight_layout()
    plt.show()


with torch.no_grad():
    _fden    = FDEN().to(DEVICE)
    _mu, _s2 = _fden(_bev)
    print(f"FDEN mu     : {_mu.shape}")
    print(f"FDEN sigma2 : {_s2.shape}")

visualise_uncertainty_map(_s2[:1], title='FDEN Demo — Random BEV (Mock Data)')

## 5. PCTAM — Probabilistic Cross-Temporal Attention Module

Temporal fusion where attention weights are **modulated by per-frame uncertainty** (Eq. 7–8).

In [ ]:
class PCTAM(nn.Module):
    """
    Probabilistic Cross-Temporal Attention Module (§III-D).

    Fuses T BEV frames, down-weighting frames with high uncertainty.

    Args:
        channels : BEV feature channels C
        n_heads  : multi-head attention heads
        tau      : temperature for attention softmax
    """
    def __init__(self, channels=BEV_C, n_heads=8, tau=1.0):
        super().__init__()
        self.tau    = tau
        self.n_heads = n_heads
        self.qkv_dim = channels

        # Q from current frame, K/V from all frames
        self.q_proj  = nn.Conv2d(channels, channels, 1)
        self.k_proj  = nn.Conv2d(channels, channels, 1)
        self.v_proj  = nn.Conv2d(channels, channels, 1)
        self.out_proj = nn.Conv2d(channels, channels, 1)

        self.norm    = nn.LayerNorm(channels)
        self.ffn     = nn.Sequential(
            nn.Conv2d(channels, channels * 2, 1),
            nn.GELU(),
            nn.Conv2d(channels * 2, channels, 1),
        )

    def forward(self, bev_seq, sigma2_seq):
        """
        bev_seq   : (B, T, C, H, W)  — temporal BEV features
        sigma2_seq: (B, T, 1, H, W)  — per-frame uncertainty maps

        Returns: fused_bev (B, C, H, W)
        """
        B, T, C, H, W = bev_seq.shape

        # Per-frame mean uncertainty: σ̄_i = mean over (H,W)
        # Shape: (B, T)  ← mean of the 1×H×W map
        sigma_mean = sigma2_seq.mean(dim=[2, 3, 4])   # (B, T)

        # Query = current frame (last), Key/Value = all frames
        q_frame = bev_seq[:, -1]                     # (B, C, H, W) — current

        # Flatten spatial dims for attention
        q = self.q_proj(q_frame).view(B, C, -1).permute(0, 2, 1)  # (B, HW, C)

        # Compute uncertainty-modulated attention over temporal frames
        attn_weights_list = []
        v_list            = []

        for t_idx in range(T):
            k_t = self.k_proj(bev_seq[:, t_idx]).view(B, C, -1).permute(0, 2, 1)  # (B, HW, C)
            v_t = self.v_proj(bev_seq[:, t_idx]).view(B, C, -1).permute(0, 2, 1)  # (B, HW, C)

            # Raw attention score: QK^T / τ
            raw_score = (q * k_t).sum(-1, keepdim=True) / self.tau  # (B, HW, 1)

            # Uncertainty penalty: exp(-σ̄_t)  — Eq. 7
            unc_penalty = torch.exp(-sigma_mean[:, t_idx]).view(B, 1, 1)  # (B, 1, 1)

            attn_weights_list.append(raw_score + torch.log(unc_penalty + 1e-8))
            v_list.append(v_t)

        # Stack across time and apply softmax — Eq. 7 normalisation
        attn_stack = torch.cat(attn_weights_list, dim=-1)     # (B, HW, T)
        attn_norm  = F.softmax(attn_stack, dim=-1)            # (B, HW, T)

        # Weighted sum of values — Eq. 8
        v_stack  = torch.stack(v_list, dim=2)                 # (B, HW, T, C)
        fused    = (attn_norm.unsqueeze(-1) * v_stack).sum(2) # (B, HW, C)

        # Reshape back to spatial
        fused = fused.permute(0, 2, 1).view(B, C, H, W)      # (B, C, H, W)
        fused = self.out_proj(fused)

        # Residual + FFN
        fused = fused + q_frame
        fused = fused + self.ffn(fused)

        return fused, attn_norm.mean(1).view(B, T)   # also return per-frame weights


# Visualise temporal attention weights
def visualise_temporal_attention(attn_weights, T=TEMPORAL_T+1):
    """
    attn_weights: (B, T) — mean attention weight per frame
    """
    w = attn_weights[0].detach().cpu().numpy()
    frame_labels = [f't-{T-1-i}' if i < T-1 else 't (current)' for i in range(T)]
    colours = ['#d73027' if v < 0.25 else '#fee090' if v < 0.5 else '#1a9850'
               for v in w]
    fig, ax = plt.subplots(figsize=(7, 3))
    bars = ax.bar(frame_labels, w, color=colours, edgecolor='black', linewidth=0.8)
    ax.set_ylabel('Attention Weight (A_i)')
    ax.set_title('PCTAM — Uncertainty-Guided Temporal Attention Weights')
    ax.set_ylim(0, 1)
    for bar, val in zip(bars, w):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{val:.3f}', ha='center', va='bottom', fontsize=9)
    plt.tight_layout()
    plt.show()


with torch.no_grad():
    _pctam     = PCTAM().to(DEVICE)
    # Simulate temporal BEV stack: (B, T, C, H, W)
    _bev_seq   = torch.randn(2, TEMPORAL_T+1, BEV_C, BEV_H, BEV_W).to(DEVICE)
    _sigma_seq = torch.rand(2, TEMPORAL_T+1, 1, BEV_H, BEV_W).to(DEVICE)
    _fused, _attn_w = _pctam(_bev_seq, _sigma_seq)
    print(f"PCTAM fused BEV: {_fused.shape}")
    print(f"PCTAM attn wts : {_attn_w.shape}")

visualise_temporal_attention(_attn_w)

## 6. EDD — Evidential Detection Decoder

Based on Evidential Deep Learning (Sensoy et al., NeurIPS 2018).  
Outputs **3D bounding boxes + aleatoric σ²** and **Dirichlet evidence → epistemic uncertainty** (Eq. 9–13).

In [ ]:
class EDD(nn.Module):
    """
    Evidential Detection Decoder (§III-E).

    From fused BEV feature map, predicts per-cell:
      - 3D bbox: (x, y, z, w, l, h, sin_r, cos_r) — 8-dim Gaussian mean + variance
      - Class:   Dirichlet evidence α_k for K classes
    """
    def __init__(self, channels=BEV_C, n_classes=NUM_CLASSES,
                 box_dim=8, n_anchors=2):
        super().__init__()
        self.n_classes = n_classes
        self.n_anchors = n_anchors
        self.box_dim   = box_dim

        # Shared neck
        self.neck = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1),
            nn.BatchNorm2d(channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, 3, padding=1),
            nn.BatchNorm2d(channels),
            nn.ReLU(inplace=True),
        )
        # Regression head — predicts (μ, σ²) per anchor per box_dim
        self.reg_mu  = nn.Conv2d(channels, n_anchors * box_dim, 1)
        self.reg_var = nn.Sequential(
            nn.Conv2d(channels, n_anchors * box_dim, 1),
            nn.Softplus()   # σ² > 0
        )
        # Evidential classification head — evidence α_k ≥ 0
        self.cls_evidence = nn.Sequential(
            nn.Conv2d(channels, n_anchors * n_classes, 1),
            nn.Softplus()   # evidence ≥ 0
        )

    def forward(self, bev_feat):
        """
        bev_feat: (B, C, H, W)
        Returns dict:
          box_mu       : (B, n_anchors, H, W, box_dim)  — predicted box params
          box_var      : (B, n_anchors, H, W, box_dim)  — localization uncertainty
          cls_evidence : (B, n_anchors, H, W, K)        — Dirichlet evidence
          cls_prob     : (B, n_anchors, H, W, K)        — normalised probabilities
          total_evidence: (B, n_anchors, H, W)          — S_i = Σ α_ik
          epistemic_unc : (B, n_anchors, H, W)          — 1/S_i
        """
        B, C, H, W = bev_feat.shape
        feat = self.neck(bev_feat)

        A, K, D = self.n_anchors, self.n_classes, self.box_dim

        # --- Regression (Eq. 10) ---
        mu  = self.reg_mu(feat).view(B, A, D, H, W).permute(0,1,3,4,2)   # (B,A,H,W,D)
        var = self.reg_var(feat).view(B, A, D, H, W).permute(0,1,3,4,2)  # (B,A,H,W,D)

        # --- Classification via Dirichlet (Eq. 11–13) ---
        alpha = self.cls_evidence(feat).view(B, A, K, H, W).permute(0,1,3,4,2)  # (B,A,H,W,K)
        S     = alpha.sum(dim=-1, keepdim=True)   # (B,A,H,W,1)  ← total evidence
        prob  = alpha / S                          # normalized class probabilities

        return {
            'box_mu'        : mu,
            'box_var'       : var,
            'cls_evidence'  : alpha,
            'cls_prob'      : prob,
            'total_evidence': S.squeeze(-1),
            'epistemic_unc' : 1.0 / (S.squeeze(-1) + 1e-6),
        }


def visualise_detections(detections, idx=0, n_top=10):
    """
    Visualise top-N detections on BEV grid.
    Green box = high evidence (low uncertainty), Red = low evidence.
    """
    prob     = detections['cls_prob'][idx]        # (A, H, W, K)
    epi_unc  = detections['epistemic_unc'][idx]   # (A, H, W)
    scores   = prob.max(-1).values                # (A, H, W)
    pred_cls = prob.argmax(-1)                    # (A, H, W)

    # Flatten and pick top-N scoring cells
    flat_scores = scores.reshape(-1).detach().cpu().numpy()
    flat_unc    = epi_unc.reshape(-1).detach().cpu().numpy()
    flat_cls    = pred_cls.reshape(-1).detach().cpu().numpy()
    top_idxs    = flat_scores.argsort()[::-1][:n_top]

    fig, ax = plt.subplots(figsize=(7, 7))
    ax.set_facecolor('#1a1a2e')
    ax.set_xlim(0, BEV_W)
    ax.set_ylim(0, BEV_H)
    ax.set_title('EDD — BEV Detections (Green=Low Unc., Red=High Unc.)')
    ax.set_xlabel('BEV Width')
    ax.set_ylabel('BEV Height')

    A_size = BEV_H * BEV_W
    for idx_ in top_idxs:
        a_idx = idx_ // A_size
        hw    = idx_ % A_size
        h_idx = hw // BEV_W
        w_idx = hw % BEV_W
        unc   = flat_unc[idx_]
        cls_  = int(flat_cls[idx_])
        color = '#2ecc71' if unc < 0.3 else '#f39c12' if unc < 0.6 else '#e74c3c'
        style = 'solid' if unc < 0.3 else 'dashed'
        rect  = patches.Rectangle((w_idx-1, h_idx-1), 2, 2,
                                   linewidth=2, edgecolor=color,
                                   linestyle=style, facecolor='none')
        ax.add_patch(rect)
        cls_name = NUSCENES_CLASSES[cls_] if cls_ < len(NUSCENES_CLASSES) else '?'
        ax.text(w_idx, h_idx, cls_name[:3], color=color,
                ha='center', va='center', fontsize=7)

    plt.tight_layout()
    plt.show()


with torch.no_grad():
    _edd   = EDD().to(DEVICE)
    _dets  = _edd(_fused)
    print("EDD outputs:")
    for k, v in _dets.items():
        print(f"  {k:20s}: {v.shape}")

visualise_detections(_dets)

## 7. Loss Functions (§III-E-2)

In [ ]:
def gaussian_nll_loss(mu, var, target):
    """
    Uncertainty-aware regression loss — Eq. 14.
    L_reg = 0.5/σ² * ||y - μ||² + 0.5 * log(σ²)
    """
    loss = 0.5 * (target - mu).pow(2) / (var + 1e-6) + 0.5 * (var + 1e-6).log()
    return loss.mean()


def evidential_classification_loss(alpha, target_class, n_classes=NUM_CLASSES):
    """
    Evidential classification loss (NeurIPS 2018, Sensoy et al.).
    Maximises evidence for correct class, penalises misleading evidence.

    alpha       : (N, K) Dirichlet parameters
    target_class: (N,)   integer class indices
    """
    # One-hot
    y = F.one_hot(target_class, n_classes).float()
    S = alpha.sum(dim=-1, keepdim=True)              # total evidence
    p = alpha / S                                     # class probs

    # Cross-entropy term
    ce_loss = (y * (torch.digamma(S) - torch.digamma(alpha))).sum(-1).mean()

    # KL regularisation: penalise evidence for wrong classes
    alpha_tilde = y + (1 - y) * alpha
    ones  = torch.ones_like(alpha_tilde)
    S_    = alpha_tilde.sum(-1, keepdim=True)
    kl    = (torch.lgamma(S_) - torch.lgamma(ones.sum(-1, keepdim=True))
             - torch.lgamma(alpha_tilde).sum(-1, keepdim=True)
             + ((alpha_tilde - 1) * (torch.digamma(alpha_tilde)
                - torch.digamma(S_))).sum(-1, keepdim=True)).mean()

    return ce_loss + 0.1 * kl


def fden_uncertainty_regularisation(sigma2):
    """
    Encourage FDEN not to predict trivially high or trivially low variance.
    """
    return (sigma2.log() + 1.0 / (sigma2 + 1e-6)).mean()


class UncertaintyAwareLoss(nn.Module):
    def __init__(self, lambda_reg=1.0, lambda_cls=1.0, lambda_unc=0.1):
        super().__init__()
        self.lambda_reg = lambda_reg
        self.lambda_cls = lambda_cls
        self.lambda_unc = lambda_unc

    def forward(self, detections, sigma2, gt_boxes, gt_labels):
        """
        Simplified assignment: match top anchor to GT boxes (nearest centroid).
        In practice use Hungarian matching (e.g. scipy.optimize.linear_sum_assignment).
        """
        B = detections['box_mu'].shape[0]
        losses = {'reg': 0., 'cls': 0., 'unc': 0.}

        for b in range(B):
            boxes  = gt_boxes[b].to(DEVICE)    # (N, 10)
            labels = gt_labels[b].to(DEVICE)   # (N,)
            N = boxes.shape[0]
            if N == 0:
                continue

            # -- Regression loss (use first anchor, centre cell, first N objects) --
            mu_flat  = detections['box_mu'][b, 0].reshape(-1, 8)[:N]    # (N, 8)
            var_flat = detections['box_var'][b, 0].reshape(-1, 8)[:N]   # (N, 8)
            box_tgt  = boxes[:, :8]                                      # (N, 8)
            if mu_flat.shape[0] > 0 and box_tgt.shape[0] > 0:
                n = min(mu_flat.shape[0], box_tgt.shape[0])
                losses['reg'] += gaussian_nll_loss(
                    mu_flat[:n], var_flat[:n], box_tgt[:n])

            # -- Evidential classification loss --
            alpha_flat = detections['cls_evidence'][b, 0].reshape(-1, NUM_CLASSES)[:N]
            if alpha_flat.shape[0] > 0:
                n = min(alpha_flat.shape[0], labels.shape[0])
                losses['cls'] += evidential_classification_loss(
                    alpha_flat[:n], labels[:n])

        # -- FDEN regularisation --
        losses['unc'] = fden_uncertainty_regularisation(sigma2)

        total = (self.lambda_reg * losses['reg']
                 + self.lambda_cls * losses['cls']
                 + self.lambda_unc * losses['unc'])
        return total, losses


print("Loss functions defined ✓")

## 8. Full Model — Uncertainty-Aware Detection Pipeline

In [ ]:
class UncertaintyAwareDetector(nn.Module):
    """
    End-to-end uncertainty-aware 3D object detection framework.

    Pipeline (Fig. 1 of paper):
      Multi-view images → Backbone → LiftSplatBEV → FDEN → PCTAM → EDD
    """
    def __init__(self):
        super().__init__()
        self.backbone  = ImageBackbone(out_channels=BEV_C)
        self.bev_enc   = LiftSplatBEV(in_channels=BEV_C,
                                       bev_h=BEV_H, bev_w=BEV_W)
        self.fden      = FDEN(channels=BEV_C)
        self.pctam     = PCTAM(channels=BEV_C)
        self.edd       = EDD(channels=BEV_C, n_classes=NUM_CLASSES)

    def forward(self, images):
        """
        images : (B, T, N_cam, 3, H, W)

        Returns:
          detections  : dict from EDD
          sigma2_stack: (B, T, 1, bev_h, bev_w) — per-frame uncertainty
          attn_weights: (B, T) — temporal attention weights
        """
        B, T, N, C, H, W = images.shape

        # --- BEV extraction for each timestep ---
        bev_list, sigma2_list = [], []
        for t in range(T):
            frame = images[:, t]                           # (B, N, 3, H, W)
            flat  = frame.view(B * N, C, H, W)            # (BN, 3, H, W)
            feat  = self.backbone(flat)                    # (BN, BEV_C, h, w)
            h, w  = feat.shape[-2:]
            feat  = feat.view(B, N, BEV_C, h, w)
            bev   = self.bev_enc(feat)                     # (B, BEV_C, bev_h, bev_w)
            mu, sigma2 = self.fden(bev)
            bev_list.append(mu)
            sigma2_list.append(sigma2)

        # --- Temporal stack ---
        bev_seq    = torch.stack(bev_list,    dim=1)      # (B, T, C, bev_h, bev_w)
        sigma2_seq = torch.stack(sigma2_list, dim=1)      # (B, T, 1, bev_h, bev_w)

        # --- PCTAM fusion ---
        fused, attn_w = self.pctam(bev_seq, sigma2_seq)   # (B, C, H, W)

        # --- EDD decoding ---
        detections = self.edd(fused)

        return detections, sigma2_seq, attn_w


model     = UncertaintyAwareDetector().to(DEVICE)
criterion = UncertaintyAwareLoss()

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total trainable parameters: {total_params:,}")

# Dry-run forward pass
with torch.no_grad():
    dummy_imgs = torch.randn(1, TEMPORAL_T+1, NUM_CAMERAS, 3, IMG_H, IMG_W).to(DEVICE)
    dets, sig2, attn = model(dummy_imgs)
    print("\nForward pass outputs:")
    for k, v in dets.items():
        print(f"  {k:22s}: {v.shape}")
    print(f"  {'sigma2_stack':22s}: {sig2.shape}")
    print(f"  {'attn_weights':22s}: {attn.shape}")

## 9. Training Loop

In [ ]:
optimizer  = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler  = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

history = {'loss': [], 'reg': [], 'cls': [], 'unc': []}

if TRAIN:
    print(f"Starting training for {NUM_EPOCHS} epochs on {DEVICE}...")
    model.train()

    for epoch in range(NUM_EPOCHS):
        epoch_loss = 0.
        n_batches  = 0

        for batch in dataloader:
            # Each batch is a list of dicts (variable-length GT boxes)
            imgs_list   = [b['images'] for b in batch]
            boxes_list  = [b['boxes'][-1] for b in batch]   # current frame GT
            labels_list = [b['labels'][-1] for b in batch]

            # Pad and stack images — all same shape
            imgs = torch.stack(imgs_list).to(DEVICE)  # (B, T+1, N, 3, H, W)

            optimizer.zero_grad()
            dets, sigma2, attn_w = model(imgs)

            sigma2_cur = sigma2[:, -1]  # current frame uncertainty
            loss, sub  = criterion(dets, sigma2_cur, boxes_list, labels_list)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            epoch_loss += loss.item()
            n_batches  += 1

        scheduler.step()
        avg = epoch_loss / max(n_batches, 1)
        history['loss'].append(avg)
        print(f"Epoch [{epoch+1:02d}/{NUM_EPOCHS}] | Loss: {avg:.4f} | "
              f"LR: {scheduler.get_last_lr()[0]:.2e}")

    print("\nTraining complete ✓")
else:
    print("TRAIN=False — skipping training.")
    print("Set TRAIN=True in cell 1 and re-run to train.")

## 10. Inference & Full Visualisation Pipeline

In [ ]:
def run_inference(model, sample, device=DEVICE):
    """
    Run a single sample through the full pipeline and visualise:
      1. FDEN uncertainty maps per temporal frame
      2. PCTAM temporal attention weights
      3. EDD detections on BEV grid
      4. Epistemic uncertainty heatmap
    """
    model.eval()
    imgs = sample['images'].unsqueeze(0).to(device)  # (1, T+1, N, 3, H, W)

    with torch.no_grad():
        dets, sigma2_seq, attn_w = model(imgs)

    T = sigma2_seq.shape[1]

    # ── 1. FDEN uncertainty across frames ───────────────────────────────────
    fig, axes = plt.subplots(1, T, figsize=(4*T, 3.5))
    if T == 1:
        axes = [axes]
    fig.suptitle('FDEN — Aleatoric Uncertainty Maps (σ²) Per Frame', fontsize=13)
    for t_idx, ax in enumerate(axes):
        umap = sigma2_seq[0, t_idx, 0].cpu().numpy()
        im   = ax.imshow(umap, cmap='hot', vmin=0)
        plt.colorbar(im, ax=ax, fraction=0.046)
        label = f't-{T-1-t_idx}' if t_idx < T-1 else 't (current)'
        ax.set_title(label)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

    # ── 2. PCTAM attention weights ───────────────────────────────────────────
    visualise_temporal_attention(attn_w, T=T)

    # ── 3. BEV detection grid ────────────────────────────────────────────────
    visualise_detections(dets)

    # ── 4. Epistemic uncertainty heatmap ────────────────────────────────────
    epi = dets['epistemic_unc'][0].mean(0).cpu().numpy()  # (H, W)
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(epi, cmap='plasma')
    plt.colorbar(im, ax=ax, label='Epistemic Uncertainty (1/S_i)')
    ax.set_title('EDD — Epistemic Uncertainty Heatmap (BEV)')
    ax.set_xlabel('BEV Width')
    ax.set_ylabel('BEV Height')
    plt.tight_layout()
    plt.show()

    print("\nSummary Statistics:")
    print(f"  Mean aleatoric σ² (current frame): {sigma2_seq[0,-1].mean().item():.4f}")
    print(f"  Mean epistemic uncertainty        : {epi.mean():.4f}")
    print(f"  Temporal attention weights        : "
          + ', '.join([f'{w:.3f}' for w in attn_w[0].cpu().numpy()]))


# Run on first sample from dataset
sample = dataset[0]
run_inference(model, sample)

## 11. Corruption Robustness Simulation

Simulates the adverse-condition evaluation from **Table II** (rain, fog, night, motion blur).

In [ ]:
class CorruptionAugmentor:
    """Applies synthetic corruptions to camera images for robustness evaluation."""

    @staticmethod
    def rain(imgs, severity=0.3):
        """Add rain-like vertical noise streaks."""
        B, T, N, C, H, W = imgs.shape
        streaks = torch.zeros_like(imgs)
        n_drops = int(W * severity * 5)
        for _ in range(n_drops):
            x = torch.randint(0, W, (1,)).item()
            length = torch.randint(H//8, H//3, (1,)).item()
            y0 = torch.randint(0, H - length, (1,)).item()
            streaks[:, :, :, :, y0:y0+length, x] = severity
        return (imgs + streaks).clamp(-3, 3)

    @staticmethod
    def fog(imgs, severity=0.4):
        """Additive Gaussian fog."""
        fog_mask = torch.randn_like(imgs) * severity * 0.3 + severity * 0.5
        return (imgs * (1 - severity * 0.3) + fog_mask).clamp(-3, 3)

    @staticmethod
    def night(imgs, severity=0.6):
        """Brightness reduction to simulate nighttime."""
        return (imgs * (1 - severity)).clamp(-3, 3)

    @staticmethod
    def motion_blur(imgs, severity=0.3):
        """Horizontal motion blur via 1D convolution."""
        ksize = max(3, int(severity * 15))
        ksize = ksize if ksize % 2 == 1 else ksize + 1
        kernel = torch.ones(1, 1, 1, ksize) / ksize
        B, T, N, C, H, W = imgs.shape
        flat = imgs.view(B*T*N*C, 1, H, W)
        blurred = F.conv2d(flat, kernel.to(imgs.device),
                           padding=(0, ksize//2))
        return blurred.view(B, T, N, C, H, W)


def evaluate_under_corruptions(model, dataset, n_samples=5):
    """
    Evaluate mean epistemic and aleatoric uncertainty under different corruptions.
    Higher uncertainty under degraded conditions = better calibration.
    """
    model.eval()
    augmentor   = CorruptionAugmentor()
    conditions  = {
        'Clean'      : lambda x: x,
        'Rain'       : augmentor.rain,
        'Fog'        : augmentor.fog,
        'Night'      : augmentor.night,
        'Motion Blur': augmentor.motion_blur,
    }
    results = {cond: {'aleatoric': [], 'epistemic': []} for cond in conditions}

    for i in range(min(n_samples, len(dataset))):
        sample = dataset[i]
        base   = sample['images'].unsqueeze(0).to(DEVICE)

        for cond_name, corrupt_fn in conditions.items():
            imgs = corrupt_fn(base.clone())
            with torch.no_grad():
                dets, sigma2_seq, _ = model(imgs)
            results[cond_name]['aleatoric'].append(
                sigma2_seq[:, -1].mean().item())
            results[cond_name]['epistemic'].append(
                dets['epistemic_unc'].mean().item())

    # Plot
    conds     = list(conditions.keys())
    ale_means = [np.mean(results[c]['aleatoric']) for c in conds]
    epi_means = [np.mean(results[c]['epistemic']) for c in conds]

    x = np.arange(len(conds))
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    ax1.bar(x, ale_means, color='steelblue', edgecolor='black')
    ax1.set_xticks(x); ax1.set_xticklabels(conds, rotation=20)
    ax1.set_title('Aleatoric Uncertainty (σ²) under Corruptions')
    ax1.set_ylabel('Mean σ²')

    ax2.bar(x, epi_means, color='coral', edgecolor='black')
    ax2.set_xticks(x); ax2.set_xticklabels(conds, rotation=20)
    ax2.set_title('Epistemic Uncertainty (1/S) under Corruptions')
    ax2.set_ylabel('Mean 1/S')

    plt.suptitle('Uncertainty Response to Adverse Conditions (Table II analogue)',
                 fontsize=12)
    plt.tight_layout()
    plt.show()

    print("\nNumerical Results:")
    print(f"{'Condition':<15} {'Aleatoric σ²':>14} {'Epistemic 1/S':>15}")
    print("-" * 46)
    for c, a, e in zip(conds, ale_means, epi_means):
        print(f"{c:<15} {a:>14.4f} {e:>15.4f}")


evaluate_under_corruptions(model, dataset, n_samples=5)

## 12. Ablation Study — Component Contributions (Table III analogue)

In [ ]:
class BaselineDetector(nn.Module):
    """Deterministic baseline — no FDEN, no PCTAM uncertainty, no EDD."""
    def __init__(self):
        super().__init__()
        self.backbone = ImageBackbone(out_channels=BEV_C)
        self.bev_enc  = LiftSplatBEV()
        self.det_head = nn.Conv2d(BEV_C, NUM_CLASSES + 8, 1)

    def forward(self, images):
        B, T, N, C, H, W = images.shape
        frame = images[:, -1]                          # only current frame
        flat  = frame.view(B*N, C, H, W)
        feat  = self.backbone(flat).view(B, N, BEV_C, H//8, W//8)
        bev   = self.bev_enc(feat)
        out   = self.det_head(bev)
        return out


class DetectorWithFDEN(nn.Module):
    """Baseline + FDEN only (no PCTAM, no EDD)."""
    def __init__(self):
        super().__init__()
        self.backbone = ImageBackbone()
        self.bev_enc  = LiftSplatBEV()
        self.fden     = FDEN()
        self.det_head = nn.Conv2d(BEV_C, NUM_CLASSES + 8, 1)

    def forward(self, images):
        B, T, N, C, H, W = images.shape
        frame = images[:, -1]
        flat  = frame.view(B*N, C, H, W)
        feat  = self.backbone(flat).view(B, N, BEV_C, H//8, W//8)
        bev   = self.bev_enc(feat)
        mu, _ = self.fden(bev)
        return self.det_head(mu)


def count_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)


def measure_inference_time(model, device=DEVICE, n_runs=5):
    dummy = torch.randn(1, TEMPORAL_T+1, NUM_CAMERAS, 3, IMG_H, IMG_W).to(device)
    model.eval()
    times = []
    with torch.no_grad():
        for _ in range(n_runs):
            start = torch.cuda.Event(enable_timing=True) if device=='cuda' else None
            end   = torch.cuda.Event(enable_timing=True) if device=='cuda' else None
            import time
            t0 = time.perf_counter()
            _ = model(dummy)
            t1 = time.perf_counter()
            times.append((t1 - t0) * 1000)
    return np.mean(times)


ablation_configs = {
    'Baseline (No Uncertainty)': BaselineDetector(),
    '+ FDEN'                   : DetectorWithFDEN(),
    '+ FDEN + PCTAM + EDD\n(Full Model)': model,
}

print(f"{'Configuration':<35} {'Params':>10} {'Latency (ms)':>14}")
print('-' * 62)
for name, m in ablation_configs.items():
    m.to(DEVICE)
    params   = count_params(m)
    latency  = measure_inference_time(m)
    print(f"{name.replace(chr(10),' '):<35} {params:>10,} {latency:>13.1f}")

# Bar chart matching Table III style
labels = ['Baseline', '+FDEN', '+FDEN+PCTAM', 'Full (+EDD)']
# Paper values from Table III
mAP_paper = [42.1, 44.3, 45.6, 46.8]

fig, ax = plt.subplots(figsize=(7, 4))
colors = ['#aec7e8', '#ffbb78', '#98df8a', '#d62728']
bars = ax.bar(labels, mAP_paper, color=colors, edgecolor='black', width=0.5)
for bar, val in zip(bars, mAP_paper):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            f'{val}', ha='center', va='bottom', fontweight='bold')
ax.set_ylabel('mAP ↑ (nuScenes)')
ax.set_title('Ablation Study — Component Contributions (Table III)')
ax.set_ylim(40, 48)
ax.axhline(y=mAP_paper[0], color='grey', linestyle='--', linewidth=0.8, label='Baseline')
ax.legend()
plt.tight_layout()
plt.show()

## 13. Model Save / Load

In [ ]:
CHECKPOINT_PATH = 'uncertainty_detector.pth'

def save_model(model, path=CHECKPOINT_PATH):
    torch.save({
        'model_state_dict': model.state_dict(),
        'config': {
            'BEV_H': BEV_H, 'BEV_W': BEV_W, 'BEV_C': BEV_C,
            'NUM_CLASSES': NUM_CLASSES, 'TEMPORAL_T': TEMPORAL_T,
        }
    }, path)
    print(f"Model saved to {path}")


def load_model(path=CHECKPOINT_PATH, device=DEVICE):
    ckpt  = torch.load(path, map_location=device)
    model = UncertaintyAwareDetector().to(device)
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()
    print(f"Model loaded from {path}")
    return model


# Uncomment to save after training:
# save_model(model)
# loaded_model = load_model()

print("Save/load utilities ready ✓")
print("\n" + "="*60)
print("NOTEBOOK COMPLETE")
print("="*60)
print("""
Components implemented:
  ✓ BEV Representation (Lift-Splat) — §III-B
  ✓ FDEN  — Feature Distribution Estimation Net — §III-C
  ✓ PCTAM — Probabilistic Cross-Temporal Attention — §III-D
  ✓ EDD   — Evidential Detection Decoder — §III-E
  ✓ Uncertainty-aware loss (Gaussian NLL + Evidential) — §III-E-2
  ✓ Corruption robustness evaluation — Table II analogue
  ✓ Ablation study — Table III analogue

Next steps:
  1. Set NUSCENES_ROOT and TRAIN=True, run all cells
  2. Tune NUM_EPOCHS, BATCH_SIZE, LR
  3. Add Hungarian matching for precise GT assignment
  4. Add mAP/NDS evaluation using nuscenes-devkit EvalBoxes API
""")